# Sentiment by Version

In this notebook I explore whether user sentiment improved as Beli delivered successive updates. The raw data has too many distinct versions to analyze cleanly, most carry only a handful of reviews, so I collapsed sub-versions into their major release, leaving 9 version groups with enough reviews each to compare.

The result is counterintuitive: negativity *rose* across versions rather than falling. I test whether that increase is real and then unpack what's driving it — which turns out to say as much about who leaves reviews as about the app itself.

In [ ]:
import pandas as pd
df = pd.read_csv("data/tagged_reviews.csv")
version_df = df[df["version"].notna()].copy()
version_df.head()

*The cell below shows that most versions have only a handful of reviews, which is why I collapsed sub-versions into major releases before analyzing.*

In [ ]:
version_df.groupby(["version"]).size().reset_index(name="n_reviews")

In [ ]:
themes_long = pd.read_csv('data/themes_long.csv')

tl = themes_long.merge(
    version_df[["review_id", "version"]],
    on="review_id",
    how="inner",  
)

tl["is_negative"] = tl["sentiment"] == "negative"

tl.head()

In [ ]:
tl["version_group"] = tl["version"].astype(str).str.split(".").str[:1].str.join(".")
by_version = (tl.groupby("version_group")
                .agg(n=("sentiment", "size"),
                     pct_negative=("is_negative", "mean"))
                .reset_index())
by_version["pct_negative"] = (by_version["pct_negative"] * 100).round(1)
by_version

The line graph below shows negativity rising across app versions. This is a counterintuitive result, since I'd expect sentiment to improve as the app matures and bugs get fixed. One thing is worth checking before drawing any conclusion: whether the increase is statistically real (rather than noise from the small early-version samples).

In [ ]:
import matplotlib.pyplot as plt

plot_df = by_version.sort_values("version_group")

fig, ax1 = plt.subplots(figsize=(11, 5))

ax1.bar(plot_df["version_group"], plot_df["n"])
ax1.set_ylabel("theme mentions (n)")
ax1.set_xlabel("version")

ax2 = ax1.twinx()
ax2.plot(by_version['version_group'],by_version['pct_negative'], color="red", marker="o", lw=2)
ax2.set_ylim(0, 100)
ax2.set_ylabel("% negative", color="red")

ax1.set_title("Overall negativity across app versions")
plt.show()

*The cells below run a permutation test on whether negativity differs between Beli's earlier and later versions. I split the versions into two eras — early (1–5) and late (6–9) — and test whether the gap in negativity between them is real or could be explained by chance.*
## The Test

- **H0 (null):** the two eras have the same negativity rate; the observed gap is chance.
- **H1 (alternative):** the two eras have different negativity rates.

**Method:** a permutation test on the negativity rate. If the era labels don't matter, I can shuffle them and the gap should be near zero. Repeating that many times builds the null distribution, which I compare the real gap against.

In [ ]:
import numpy as np

tl["version_group"] = pd.to_numeric(tl["version_group"])

late_versions = [6, 7, 8, 9]

period = tl.copy()
period["era"] = "early"
period.loc[period["version_group"].isin(late_versions), "era"] = "late"

period.groupby("era")["is_negative"].agg(n="size", pct_negative="mean")

In [ ]:
rng = np.random.default_rng(42)

is_negative = period["is_negative"]
is_late = (period["era"] == "late")


def negativity_gap(tbl, late):
    """Late negativity − early negativity, in percentage points."""
    return (tbl[late].mean() - tbl[~late].mean()) * 100


def one_simulated_gap(tbl, late):
    """Shuffle the era labels once, then recompute the gap."""
    return negativity_gap(tbl, rng.permutation(late))

In [ ]:
observed_diff = negativity_gap(is_negative, is_late)
print(f"Observed difference (late − early): {observed_diff:.1f} pp")

In [ ]:
n_iter = 10000
simulated_gaps = np.array([
    one_simulated_gap(is_negative, is_late)
    for i in range(n_iter)
])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(simulated_gaps, bins=30, edgecolor="white")
ax.axvline(observed_diff, color="red", lw=2, label=f"observed ({observed_diff:.1f} pp)")
ax.set_xlabel("late − early negativity (pp), under H0")
ax.set_ylabel("frequency")
ax.set_title("Negativity gap under H0 (era labels shuffled)")
ax.legend()
plt.show()

In [ ]:
p_value = np.mean(np.abs(simulated_gaps) >= np.abs(observed_diff))
print(p_value)

The histogram is the null distribution: the gaps I'd get if era didn't matter, built by shuffling the labels 10,000 times. The red line is the gap I actually observed. It sits far outside the shuffled distribution, which is why the p-value is so small. A gap this large essentially never arises by chance.

## Permutation Results

The test confirms the negativity gap between early and late versions is significant (p < 0.0001), so it's unlikely to be sampling noise. But it doesn't explain why the gap exists. Review volume grew sharply across versions (from ~19 to ~800 reviews), which probably reflects a broadening, more critical reviewer base — and the test can't rule that out. Telling apart a genuine decline in the app from a shift in who reviews it would need evidence this test doesn't provide.

The App Store sampling is another caveat. The RSS feed only returns the most recent reviews (capped at ~500), not the full review base, so the dataset over-represents certain versions and time periods rather than sampling evenly across the app's history.

## Conclusion

The rising negativity across versions is a real effect (the permutation test rules out chance), but it's hard to attribute cleanly. The comment base differs too much between versions to compare directly: early versions have far fewer reviews, likely from a different, earlier-adopter audience than the later cohorts. Voluntary response bias may also play a role - people tend to review when they feel strongly, either positive or negative, so the sample over-represents extreme opinions rather than the typical user. The trend tells that something shifted, but not whether the app itself declined or simply who was reviewing it.

# Sentiment by Quarter (Google Play)

After the sentiment by version analysis, I wanted to look at sentiment over calendar time — partly to see whether the trend tracks when reviews were written rather than which version they target, and partly because time and version tell different stories (a time trend can pick up external events, like press or a viral moment, that aren't tied to a release).

One important limitation up front: this analysis covers **Google Play only**. The App Store RSS feed doesn't expose the date a review was written, so all 562 App Store reviews are excluded, leaving ~262 dated Google Play reviews. This is therefore the sentiment trend of Beli's *Android* reviewers, not the full user base, and the two platforms may not move together.

Because the dated reviews are sparse (often single digits per month), I bucket by **quarter** rather than month, so each point rests on enough reviews to be meaningful.

In [ ]:
df = pd.read_csv("data/themes_long.csv")
gp = df[df['source'] == 'google_play']

tr = pd.read_csv("data/tagged_reviews.csv")
tr = tr[tr['source'] == 'google_play']
dates = tr[['date', 'review_id']]

gp_dates = gp.merge(dates, on='review_id', how='right')
gp_dates

In [ ]:
gp_dates['is_negative'] = gp_dates['sentiment'] == 'negative'
gp_dates['date'] = pd.to_datetime(gp_dates["date"])
gp_dates["quarter"] = gp_dates["date"].dt.to_period("Q").astype(str)
gp_quarter = gp_dates[['source', 'is_negative', 'quarter']]
gp_quarter

In [ ]:
by_quarter = gp_quarter.groupby('quarter').agg(n=('source', 'size'), pct_negative=('is_negative', 'mean')).reset_index()
by_quarter["pct_negative"] = (by_quarter["pct_negative"] * 100).round(1)
by_quarter

In [ ]:
fig, ax1 = plt.subplots(figsize=(11,5))

ax1.bar(by_quarter["quarter"], by_quarter["n"])
ax1.set_ylabel("theme mentions (n)")
ax1.set_xlabel("quarter")

ax2 = ax1.twinx()
ax2.plot(by_quarter["quarter"], by_quarter["pct_negative"], color="red", marker="o", lw=2)
ax2.set_ylabel("% negative", color="red")
ax2.set_ylim(0, 100)

ax1.tick_params(axis="x", rotation=45)
ax1.set_title("Google Play: negativity by quarter")
plt.show()

The quarterly trend closely mirrors the version analysis: percent negative descreases initially, then rises progressively over time. This convergence is expected. Later reviews target later versions, so the two axes are largely capturing the same underlying shift rather than two independent factors.